# Arcadia: full training run on a Kaggle T4

Before running: in the notebook's **Settings** panel set **Accelerator → GPU T4** and turn **Internet on**
(the notebook clones the repo and downloads the Gutenberg books and DreamBank).

Run the cells in order. Everything is written under `/kaggle/working/`, so it is kept when you save a version.

## 1. Clone the repo

In [ ]:
# The training code lives on this branch (main only has the first scaffold commit).
REPO = "https://github.com/DanielGregory/Arcadia"
BRANCH = "claude/arcadia-ml-setup-hhn3dv"

import os
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 -b {BRANCH} {REPO} /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1

## 2. Install only what Kaggle is missing

Kaggle already has torch, numpy, pillow, requests and tqdm. Torch is **not** reinstalled.

In [ ]:
import importlib.util, subprocess, sys

missing = [pkg for pkg in ("tiktoken", "datasets") if importlib.util.find_spec(pkg) is None]
print("installing:", missing or "nothing, all present")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 3. Check the GPU

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: set Accelerator to GPU T4")

## 4. Build the corpus

Downloads the books and DreamBank, cleans them, and writes `data/arcadia.txt`, `train.txt` and `val.txt`.

In [ ]:
!python -m engine.data_prep.build_corpus

## 5. Train

Settings are in `configs/full.py`: 6 layers, 6 heads, n_embd 384, context 256, batch 64, dropout 0.2,
learning rate 3e-4 cosine-decayed to 3e-5 over 5000 steps, evaluation every 250 steps.

Checkpoints (`ckpt.pt`, `best.pt`), `loss_log.csv` and a short `sample.txt` go to `/kaggle/working/run1`.

In [ ]:
!python -m engine.train_char --config configs/full.py --out_dir /kaggle/working/run1

## 6. Sample 2000 characters at three temperatures

From the best checkpoint (lowest validation loss).

In [ ]:
!python -m engine.sample --ckpt /kaggle/working/run1/best.pt --chars 2000 --temperature 0.7 0.9 1.1

## 7. Upload the run to Hugging Face (optional)

One-time setup:
1. On huggingface.co: Settings → Access Tokens → create a token with **write** access.
2. In this notebook: **Add-ons → Secrets** → add a secret named `HF_TOKEN` with that token, and tick it for this notebook.
3. `HF_REPO` below is set to `DannyGregs/arcadia-dreams`.

The repo is created **private** on the first upload. Without the secret this cell skips the upload and changes nothing.

In [ ]:
HF_REPO = "DannyGregs/arcadia-dreams"
RUN_DIR = "/kaggle/working/run1"

import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as err:
    print("no HF_TOKEN secret, skipping upload:", err)
else:
    if HF_REPO.startswith("YOUR-"):
        print("set HF_REPO first")
    else:
        !python -m engine.hub push --repo {HF_REPO} --run_dir {RUN_DIR}